# General Dynamics Geometry Workflow

This tutorial introduces the generic finite-dimensional topology workflow in pyna.  The goal is not to learn design patterns as names, but to see how helper objects keep the mathematical steps explicit:

1. integrate a continuous flow into a `Trajectory`,
2. promote a closed trajectory into a `Cycle`,
3. cut a cycle by a section to get a `PeriodicOrbit`,
4. iterate a map into an `Orbit` and verify a `PeriodicOrbit`,
5. assemble synthetic O/X cycles into a `Tube` and cut it into an `IslandChain`.

`TopologyWorkflow` is a facade over factories, builders and bridges.  It keeps notebook code readable while still making each promotion explicit.

## Setup

The example uses only NumPy and pure Python pyna classes, so it runs without cyna or external equilibrium data.

In [ ]:
import numpy as np

from pyna.topo import TopologyWorkflow
from pyna.topo.core import Cycle, Trajectory
from pyna.topo.section import HyperplaneSection

wf = TopologyWorkflow(closure_tol=1e-3)

## 1. Flow to Trajectory

Start with the harmonic oscillator.  `DynamicalSystemFactory` is available through the workflow so the construction can later be moved into a config file or teaching exercise without changing the analysis steps.

In [ ]:
flow = wf.system(
    "callable-flow",
    rhs=lambda x, t: np.array([x[1], -x[0]]),
    dim=2,
    coordinate_names=("q", "p"),
    label="harmonic oscillator",
)

traj = wf.trajectory(flow, [1.0, 0.0], (0.0, 2.0*np.pi), dt=0.01)
print(type(traj).__name__, traj.n_samples, traj.final)
print("closing error:", wf.closing_error(traj))

A `Trajectory` is sampled geometry.  It is useful, but it is not automatically a proof of invariance.  The next step is an explicit promotion.

## 2. Trajectory to Cycle

`closed_cycle` refuses open samples.  This is the central safety rule: sampled data and invariant objects are related, but not interchangeable.

In [ ]:
cycle = wf.closed_cycle(traj)
print(type(cycle).__name__, cycle.period_value, cycle.ambient_dim)

open_traj = Trajectory(
    states=np.array([[0.0, 0.0], [1.0, 0.0]]),
    times=np.array([0.0, 1.0]),
)
try:
    wf.closed_cycle(open_traj)
except ValueError as exc:
    print("open trajectory rejected:", exc)

## 3. Section Cut: Cycle to PeriodicOrbit

A Poincare section bridges continuous-time and discrete-time geometry.  Cutting a `Cycle` produces a `PeriodicOrbit` of the return map.

In [ ]:
section = HyperplaneSection(np.array([1.0, 0.0]), 1.0, phase_dim=2, _label="q=1")
po_from_cycle = wf.section_cut(cycle, section)
print(type(po_from_cycle).__name__, "period =", po_from_cycle.period)
print("first section point:", po_from_cycle.points[0].state)

`PoincareMapFactory` chooses an executable implementation.  For this generic flow and hyperplane section, the portable `GeneralPoincareMap` backend is selected.

In [ ]:
pmap = wf.poincare_map(flow, section, dt=0.05)
print(type(pmap).__name__)

## 4. Map to Orbit to PeriodicOrbit

For maps, the same rule applies.  Iterating a map gives an `Orbit`; promoting samples to `PeriodicOrbit` can be verified against the map.

In [ ]:
flip = wf.system("callable-map", step_func=lambda x: -x, dim=2, label="flip map")
orbit = wf.orbit(flip, [1.0, 0.0], n_iter=4)
print(type(orbit).__name__, orbit.states.tolist(), "period guess =", orbit.period_guess)

period_two = wf.periodic_orbit([[1.0, 0.0], [-1.0, 0.0]], map_obj=flip, verify=True)
print(type(period_two).__name__, period_two.period)

## 5. Tube to IslandChain

A `Tube` is continuous-time resonance geometry.  A section cut turns it into an `IslandChain`.  Here the cycles are synthetic line segments so the construction is transparent.

In [ ]:
def crossing_cycle(y):
    return Cycle(
        trajectory=Trajectory(
            states=np.array([[-1.0, y], [1.0, y]]),
            times=np.array([0.0, 1.0]),
        )
    )

tube = wf.tube(crossing_cycle(0.0), [crossing_cycle(0.25)], label="teaching tube")
cut_section = HyperplaneSection(np.array([1.0, 0.0]), 0.0, phase_dim=2, _label="q=0")
islands = wf.section_cut(tube, cut_section)
print(type(islands).__name__, islands.n_islands)
print("O point:", islands.O_points[0].state)
print("X point:", islands.X_points[0].state)

## What the Helpers Are For

- `TopologyWorkflow` keeps tutorial code linear.
- Adapters normalize arrays and solver outputs.
- Builders make promotion explicit and validate closure when requested.
- Bridges express the mathematical section cut from continuous-time objects to discrete-time objects.
- Factories choose executable implementations without making users remember backend classes.

The helper layer should disappear into the workflow when it is useful; it should not force users to think about design-pattern terminology.